In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv

In [13]:
load_dotenv()

True

In [14]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",      
    temperature=0.7,
)

In [15]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [16]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']
    prompt = f'Generate a detailed outline for a blog on {title}'
    outline = model.invoke(prompt).content
    state['outline'] = outline
    return state

In [17]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f'Write a detailed blog on the {title} using following outline - {outline}'
    content = model.invoke(prompt).content
    state['content'] = content
    return state

In [18]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [19]:
final_state = workflow.invoke({'title': 'Situation of AI in Pakistan'})
print(final_state)

{'title': 'Situation of AI in Pakistan', 'outline': 'Here\'s a detailed outline for a blog post on the "Situation of AI in Pakistan," designed to be informative, engaging, and comprehensive.\n\n---\n\n## Blog Title Options:\n\n*   **Pakistan\'s AI Awakening: Navigating the Future of Artificial Intelligence**\n*   **The AI Landscape in Pakistan: Opportunities, Challenges, and the Road Ahead**\n*   **Unlocking Potential: The Current State of AI in Pakistan**\n*   **From Code to Impact: Understanding AI\'s Journey in Pakistan**\n\n---\n\n## I. Introduction (Approx. 150-200 words)\n\n*   **A. Hook:** Start with a compelling statement about the global AI revolution and its transformative power.\n    *   *Example:* "Artificial Intelligence is not just a buzzword; it\'s the defining technology of our era, reshaping industries, economies, and societies worldwide."\n*   **B. Contextualize for Pakistan:** Briefly introduce Pakistan\'s position in this global shift.\n    *   *Example:* "As nation